Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Master Course #24512

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Exercise: 2D Convolution Torch vs. Manual

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(5)

## Input Image

In [ ]:
X = rng.integers(-128, 127, (64, 32))
# set up a face shape within the noise
X[10:15, 10:15] = -128
X[12:16, 25:30] = -128
X[20:28, 18:21] = 127
X[35:37, 12:25] = 127
if False:  # use a smaller test case
    X = rng.integers(-128, 127, (8, 4))
Xt = torch.Tensor(X[None, None, :])
X, Xt, X.shape, Xt.shape

In [ ]:
plt.imshow(X, cmap='gray')

# Kernel / 2D Filter

In [ ]:
Nrow, Ncol = 5, 3
kernel = rng.integers(-128, 127, (Nrow, Ncol)) * 0  # zero matrix as integer
kernel[0:3, 0] = 1  # white block at upper left edge
kernel[3:5, -1] = -1  # black block at lower right edge

# torch calculates correlation, so pre-flip kernel:
kernel_flipped = np.copy(np.fliplr(np.flipud(kernel)))
kernelt = torch.Tensor(kernel_flipped[None, None, :])
kernel, kernelt, kernel.shape, kernelt.shape

In [ ]:
plt.imshow(kernel, cmap='gray')

## Torch Conv2D

In [ ]:
# torch convolution (which is actually a correlation)
f = torch.nn.Conv2d(in_channels=1,
                    out_channels=1,
                    kernel_size=(Nrow, Ncol),
                    padding = 'same',
                    padding_mode = 'zeros',
                    bias=False)
f.weight = torch.nn.Parameter(kernelt)  # we use the flipped kernel
# to calculate the intended 2D convolution

# padding same -> the output image has the same size as the input image

f.weight  # we see the flipped kernel: black at upper left, white at lower right

In [ ]:
Yt = f(Xt)  # do the Conv2d operation
Y = Yt[0, 0, :, :].detach().numpy()
Yt.shape, Y.shape

In [ ]:
# output image
plt.imshow(Y, cmap='gray')

## Manual 2D Convolution

In [ ]:
# prep tmp image for padding = 'same', padding_mode = 'zeros'
Xtmp = np.zeros((2*(Nrow//2)+X.shape[0],
                 2*(Ncol//2)+X.shape[1]))
Xtmp[Nrow//2:-Nrow//2+1,
     Ncol//2:-Ncol//2+1] = X
Ytmp = np.zeros_like(Xtmp)
plt.imshow(Xtmp, cmap='gray')

In [ ]:
# arrays for the pixels at which the convolution
# is calculated
r = np.arange(Nrow//2, Nrow//2+X.shape[0])
c = np.arange(Ncol//2, Ncol//2+X.shape[1])

kernel_flipped = np.flipud(np.fliplr(kernel))

# do for all these pixels
for ri in r: 
    for ci in c:
        # grab image slice according to kernel size: 
        tmp = Xtmp[ri-Nrow//2:ri+Nrow//2+1,
                   ci-Ncol//2:ci+Ncol//2+1]
        # this is the actual convolution, i.e. 
        # multiply the corresponding image and kernel pixels
        # and then double sum over rows and columns
        Ytmp[ri, ci] = np.sum(kernel_flipped*tmp, axis=None)

        
Ymanual = Ytmp[Nrow//2:-1-Nrow//2+1,
               Ncol//2:-1-Ncol//2+1]  # cut out the actual result image
Ymanual.shape

In [ ]:
plt.imshow(Ymanual, cmap='gray')

In [ ]:
plt.subplot(1,2,1)
plt.imshow(Y, cmap='gray')
plt.title('torch')
plt.subplot(1,2,2)
plt.imshow(Ymanual, cmap='gray')
plt.title('manual')

np.allclose(Y, Ymanual)

# looks like a heavily zoomed in photo of the Marsian surface, right? :)

In [ ]:
# same stuff as above into a function
def my_conv2D_same_zero(X, kernel):
     Nrow, Ncol = kernel.shape[0], kernel.shape[1]

     # prep tmp image for padding = 'same', padding_mode = 'zeros'
     Xtmp = np.zeros((2*(Nrow//2)+X.shape[0],
                      2*(Ncol//2)+X.shape[1]))
     Xtmp[Nrow//2:-Nrow//2+1,
          Ncol//2:-Ncol//2+1] = X
     Ytmp = np.zeros_like(Xtmp)
     kernel_flipped = np.flipud(np.fliplr(kernel))
     
     # manual double for loop: do for all pixels
     for ri in np.arange(Nrow//2, Nrow//2+X.shape[0]): 
          for ci in np.arange(Ncol//2, Ncol//2+X.shape[1]):
               # grab image at position ri,ci according to kernel size: 
               tmp = Xtmp[ri-Nrow//2:ri+Nrow//2+1,
                         ci-Ncol//2:ci+Ncol//2+1]
               
               # this is the actual convolution, i.e. 
               # multiply the corresponding image and kernel pixels
               # and then double sum it over rows and columns
               Ytmp[ri, ci] = np.sum(kernel_flipped*tmp, axis=None)

     # cut out the actual result image
     Y = Ytmp[Nrow//2:-1-Nrow//2+1,
              Ncol//2:-1-Ncol//2+1]
     return Y

In [ ]:
np.allclose(Y, my_conv2D_same_zero(X, kernel))

In [ ]:
# Think about (coding is probably very helpful):
# - what is Conv2D doing with multiple input channels?
# - why can Conv2D have multiple output channels?
# - what are the related kernels then?

# Further, the Conv2D parameters 'stride' and 'dilation' should be really understood.

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.